# Lab 4 — The same harness with LangChain (EP09)

The book's §18.9 compares frameworks with building your own, and says LangChain has *"a rich ecosystem of integrations but … abstractions that can obscure what is actually happening"*

This lab builds Lab 1's research assistant again with LangChain's `create_agent`, and matches it piece by piece:

| Our harness (Lab 1) | LangChain |
|---|---|
| `ToolDefinition` | `@tool` (schema from type hints + docstring) |
| approval in `ToolExecutor` | `HumanInTheLoopMiddleware` + a checkpointer |
| retry + backoff | `ToolRetryMiddleware` |
| `ContextManager` | `ContextEditingMiddleware` |
| `MAX_ITERATIONS` | `ModelCallLimitMiddleware` |
| `LoopDetector` | nothing quite the same — we write it as middleware |
| log + `run_id` | `astream(..., stream_mode="updates")` + `thread_id` |

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.9–18.10 — CC BY-SA 4.0
> The tools in this lab use a simulated paper library, but every model answer comes from a real run

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs under 1 US cent) |
| Packages | `langchain`, `langchain-openai`, `langgraph`, `tokenizers` — versions pinned, because LangChain's API changes often |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

In [ ]:
%pip install -q "langchain==1.4.7" "langchain-openai==1.7.2" "langgraph==1.2.16" python-dotenv "tokenizers==0.23.3"

## 1. The model

`ChatOpenAI` replaces `AsyncOpenAI` + `client.chat.completions.create(...)` — model, provider and timeout exactly as in Lab 3

In [ ]:
import asyncio, hashlib, json, os, time, uuid
from langchain_openai import ChatOpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
model = ChatOpenAI(model=MODEL, base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                   temperature=0, max_retries=5, timeout=60,
                   extra_body={"reasoning": {"enabled": False},
                               "provider": {"order": ["makora"], "allow_fallbacks": False}})

## 2. Tools with `@tool`

In Lab 1 we wrote `ToolDefinition` with its JSON schema by hand — `@tool` builds the schema from the type hints and the docstring.
`requires_approval` moves into middleware (part 3)

In [ ]:
LIBRARY = {
    "P1": ("Informer (2021)", "ProbSparse self-attention makes long-sequence time-series forecasting faster."),
    "P2": ("PatchTST (2023)", "Splits each series into patches of 16 steps; each channel is handled on its own."),
    "P3": ("TimesFM (2024)", "A decoder-only foundation model that forecasts new series zero-shot."),
    "P4": ("Autoformer (2021)", "Replaces attention with auto-correlation and decomposes trend and season."),
    "P5": ("DLinear (2023)", "A single linear layer that beats many transformers on long-term forecasting."),
    "P6": ("BERT (2019)", "Pre-trains a bidirectional transformer on text with masked language modelling."),
}
SAVED = {}  # the simulated disk

In [ ]:
from langchain.tools import tool

@tool
async def search_papers(query: str, max_results: int = 3) -> str:
    """Search the paper library. Returns ids and titles."""
    await asyncio.sleep(0.5)
    words = [w.rstrip("s") for w in query.lower().replace("-", " ").split() if len(w) > 3]
    hits = [(pid, t) for pid, (t, a) in LIBRARY.items() if any(w in (t + " " + a).lower() for w in words)]
    return json.dumps([{"id": pid, "title": t} for pid, t in hits[:max_results]])

@tool
async def read_paper(paper_id: str) -> str:
    """Read one paper's abstract by its id (P1, P2, ...)."""
    await asyncio.sleep(1.5)
    title, abstract = LIBRARY[paper_id]
    return json.dumps({"id": paper_id, "title": title, "abstract": abstract})

@tool
def save_report(filename: str, text: str) -> str:
    """Save a finished report to a file."""
    SAVED[filename] = text
    return f"saved {len(text)} characters to {filename}"

TOOLS = [search_papers, read_paper, save_report]
print(search_papers.args)

## 3. Middleware — the harness's parts

**middleware** = code LangChain runs around every model call or tool call — each one does one job, like one class in our harness

In [ ]:
from langchain.agents.middleware import (HumanInTheLoopMiddleware, ToolRetryMiddleware,
    ModelCallLimitMiddleware, ContextEditingMiddleware, ClearToolUsesEdit, wrap_tool_call)

approval = HumanInTheLoopMiddleware(
    interrupt_on={"save_report": {"allowed_decisions": ["approve", "reject"]}})

retry = ToolRetryMiddleware(max_retries=2, initial_delay=1.0, backoff_factor=2.0)
limit = ModelCallLimitMiddleware(run_limit=50)

from langchain_core.messages.utils import count_tokens_approximately
from tokenizers import Tokenizer
tok = Tokenizer.from_pretrained(MODEL)

def gemma_tokens(messages) -> int:
    return sum(len(tok.encode(str(m.content)).ids) + 4 for m in messages)

context = ContextEditingMiddleware(token_counter=gemma_tokens,
    edits=[ClearToolUsesEdit(trigger=128_000, keep=3)])

In [ ]:
from langchain_core.messages import HumanMessage
for text in ["You are a research assistant for a university course.",
             "สวัสดีครับ วันนี้เราจะสร้าง harness ด้วย LangChain",
             '{"id": "P2", "title": "PatchTST (2023)"}']:
    msg = [HumanMessage(text)]
    print(f"approx {count_tokens_approximately(msg):3}   gemma {gemma_tokens(msg):3}   {text}")

In [ ]:
from langchain_core.messages import ToolMessage

fingerprints = []

@wrap_tool_call
async def loop_detector(request, handler):
    call = request.tool_call
    h = hashlib.md5(f"{call['name']}:{json.dumps(call['args'], sort_keys=True)}".encode()).hexdigest()
    fingerprints.append(h)
    if fingerprints[-5:].count(h) >= 2:
        return ToolMessage(f"Loop detected: {call['name']} was called again with the same arguments.",
                           tool_call_id=call["id"], name=call["name"], status="error")
    return await handler(request)

## 4. Assemble the agent and run it (approve ✅)

`create_agent` = the ReAct loop of `AgentHarness.run` · the **checkpointer** keeps the run's state, so it can stop for approval and carry on · `thread_id` names the run, like `run_id`

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

SYSTEM = ("You are a research assistant for a university course. Use the tools to find and read papers "
          "before answering. When you need several papers, request them all in the same turn. "
          "Always cite paper ids like [P2].")
TASK = ("Find papers on transformers for time-series forecasting, read the three most relevant ones, "
        "and save a short report to report.md.")

agent = create_agent(model, TOOLS, system_prompt=SYSTEM,
                     middleware=[approval, loop_detector, retry, limit, context],
                     checkpointer=InMemorySaver())

async def run(inp, config):
    async for step in agent.astream(inp, config, stream_mode="updates"):
        for node, update in step.items():
            if node == "__interrupt__":
                for req in update[0].value["action_requests"]:
                    print(f"🔒 approval needed: {req['name']}(filename={req['args'].get('filename')!r})")
                continue
            for m in (update or {}).get("messages", []):
                if getattr(m, "tool_calls", None) and node == "model":
                    print("model asks:", ", ".join(f"{c['name']}({c['args'].get('paper_id') or c['args'].get('query') or c['args'].get('filename')})" for c in m.tool_calls))
                elif m.type == "tool":
                    print(f"  {m.name} → {str(m.content)[:70]}")
                elif node == "model":
                    print("ANSWER:", m.content)

def usage(config) -> str:
    msgs = agent.get_state(config).values["messages"]
    u = [m.usage_metadata for m in msgs if getattr(m, "usage_metadata", None)]
    return f"{len(u)} model calls · {sum(x['input_tokens'] for x in u):,} in + {sum(x['output_tokens'] for x in u):,} out"

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
t0 = time.monotonic()
await run({"messages": [{"role": "user", "content": TASK}]}, config)
print(f"\n⏸ paused · saved so far: {list(SAVED)}")

In [ ]:
await run(Command(resume={"decisions": [{"type": "approve"}]}), config)
print(f"\nsaved: {list(SAVED)} · {usage(config)} · {time.monotonic() - t0:.1f}s")

## 5. Reject ❌ — what the model says, and what really happened

In [ ]:
SAVED.clear(); fingerprints.clear()
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
await run({"messages": [{"role": "user", "content": TASK}]}, config)
await run(Command(resume={"decisions": [{"type": "reject"}]}), config)
print(f"\nsaved: {list(SAVED)} · {usage(config)}")

## 6. Middleware order

The book: *"Build custom when … the framework's abstractions leak in ways that cause bugs"* — let's see a real one.
Ask for paper `P9`, which isn't in the library → `read_paper` raises `KeyError` → `ToolRetryMiddleware` retries

In [ ]:
async def ask(middleware, question):
    global agent
    agent = create_agent(model, TOOLS, system_prompt=SYSTEM, middleware=middleware,
                         checkpointer=InMemorySaver())
    fingerprints.clear()
    t0 = time.monotonic()
    await run({"messages": [{"role": "user", "content": question}]},
              {"configurable": {"thread_id": str(uuid.uuid4())}})
    print(f"({time.monotonic() - t0:.1f}s)")

await ask([approval, retry, loop_detector, limit, context], "Read paper P9 and tell me what it is about.")

In [ ]:
await ask([approval, loop_detector, retry, limit, context], "Read paper P9 and tell me what it is about.")

## Summary — piece by piece

| Our harness | LangChain | What we saw in the lab |
|---|---|---|
| `ToolDefinition` | `@tool` | the schema comes from the type hints |
| approval | `HumanInTheLoopMiddleware` | pauses the whole run, keeps its state, continues with `Command(resume=...)` |
| retry + backoff | `ToolRetryMiddleware` | three tries, the same — but the middleware order changes the result |
| `ContextManager` | `ContextEditingMiddleware` | clears old tool results instead of dropping messages · estimates tokens unless we pass a tokenizer |
| `LoopDetector` | — | written ourselves with `@wrap_tool_call` |
| `MAX_ITERATIONS` | `ModelCallLimitMiddleware` | |
| log + `run_id` | `astream` + `thread_id` | you need to know what each node sends |
| truncating output at 2,000 tokens | — | not built in — exercise 1 |

**Both ways:** after a rejection the model sometimes says it saved the report when it didn't — trust the system's state (`SAVED`), not the model's words

**Exercises**
1. Write a `@wrap_tool_call` middleware that cuts a tool result over 2,000 tokens with `tok`, like `_truncate` in Lab 1
2. Make `ToolRetryMiddleware` not retry on `KeyError` (see its `retry_on` argument) — how long does the P9 question take now?
3. Use Lab 3's `record` and `compare` on this agent — can LangChain's traces be compared with ours? Where do you get the data from?